In [ ]:
STRIPS_DIR  = '/Users/alyrajan/Desktop/Mudd/Computer Science/CS158 Project/MLProjectSp26/Data/Training/Strips'
MASKS_DIR   = '/Users/alyrajan/Desktop/Mudd/Computer Science/CS158 Project/MLProjectSp26/Data/Training/Masks'
MODEL_PATH  = '/Users/alyrajan/Desktop/Mudd/Computer Science/CS158 Project/MLProjectSp26/best_model.pt'


In [ ]:
#Took the autogenerated mask function from createMaskUNet (Aly)

import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

def generate_mask(img_bgr: np.ndarray) -> np.ndarray:
    """
    Generate a binary mask from an ECG strip image.
    """
    b, g, r = cv2.split(img_bgr)

    # Isolate near-black pixels
    black_mask = ((b < 80) & (g < 80) & (r < 80)).astype(np.uint8) * 255

    # Close small gaps in the signal, remove isolated noise
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    mask = cv2.morphologyEx(black_mask, cv2.MORPH_CLOSE, kernel_close, iterations=2)
    mask = cv2.morphologyEx(mask,       cv2.MORPH_OPEN,  kernel_open,  iterations=1)

    return mask


strip_paths = list(Path(STRIPS_DIR).glob('*.png')) + list(Path(STRIPS_DIR).glob('*.jpg'))
print(f'Found {len(strip_paths)} strips')

for strip_path in tqdm(strip_paths, desc='Generating masks'):
    img  = cv2.imread(str(strip_path))
    mask = generate_mask(img)
    out_path = Path(MASKS_DIR) / (strip_path.stem + '_mask.png')
    cv2.imwrite(str(out_path), mask)

print('Done — masks saved to', MASKS_DIR)